# EDA: Workouts + Fitbit

This notebook performs exploratory data analysis for the merged workouts and Fitbit datasets. It uses `avg_heart_rate` from `fitbit_merged.csv` as the heart-rate metric.

Files used: `data/processed_merged.csv`, `data/features_model1.csv`, `data/fitbit_merged.csv`

In [22]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
sns.set(style='whitegrid')
%matplotlib inline

In [19]:
# Paths and load
ROOT = Path('..').resolve() if Path('.').name != 'health and weight training metrics' else Path('.')
DATA = Path('data')
p_proc = '../data/processed_merged.csv'
p_feat = '../data/features_model1.csv'
p_fitbit = '../data/fitbit_merged.csv'
p_workouts = '../data/workouts.csv'
#print('paths:')
#print(p_proc.resolve())
#print(p_fitbit.resolve())

proc = pd.read_csv(p_proc, parse_dates=['date'])
feat1 = pd.read_csv(p_feat, parse_dates=['date'])
fitbit = pd.read_csv(p_fitbit, parse_dates=['date'])
workout = pd.read_csv(p_workouts)

print('shapes:')
print('processed_merged:', proc.shape)
print('features_model1:', feat1.shape)
print('fitbit_merged:', fitbit.shape)

proc.head(2)

shapes:
processed_merged: (4019, 31)
features_model1: (4019, 18)
fitbit_merged: (516, 4)


,date,exercise_title,total_volume,total_sets,avg_weight,max_weight,total_reps,best_est_1RM,rolling_best_prev,relative_strength,...,days_since_last_pr,pr_freq_90d,sessions_since_last_pr,distance_to_personal_best,sleep_minutes,sleep_7d_avg,sleep_dev_from_14d,resting_hr,hr_7d_avg,hr_baseline_z
0,2024-07-18,Air Bike,0.0,1,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-20,Back Extension (Hyperextension),0.0,2,0.0,0.0,35.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,391.571429,-384.785714,71.0,73.571429,-1.307617


In [11]:
# Ensure heart rate column is available as `avg_heart_rate` and create alias `resting_hr` for feature compatibility
if 'avg_heart_rate' in fitbit.columns:
    fitbit = fitbit.rename(columns={'avg_heart_rate':'resting_hr'})

# show sample
fitbit[['date','resting_hr','sleep_minutes','steps']]


,date,resting_hr,sleep_minutes,steps
0,2024-12-01,71.0,333,9973
1,2024-12-02,72.0,335,13465
2,2024-12-03,76.0,909,1903
3,2024-12-04,74.0,648,2732
4,2024-12-05,73.0,511,5491
...,...,...,...,...
511,2026-04-26,70.0,522,10208
512,2026-04-27,71.0,359,14506
513,2026-04-28,70.0,484,13790
514,2026-04-29,68.0,443,9068


In [ ]:
def parse_start_date(s: str): #pd.Timestamp | pd.NaT:
    """Extract the date portion from varied start_time strings and return a Timestamp (date only).

    Examples handled:
    - "May 24, 2026, 8:15p.m"
    - "May 24, 2026, 8:15 p.m."
    - other spacing / punctuation variants
    """
    if pd.isna(s):
        return pd.NaT
    # capture patterns like 'May 24, 2026' at start of string
    m = re.search(r"([A-Za-z]+\s+\d{1,2},\s*\d{4})", str(s))
    if not m:
        try:
            return pd.to_datetime(s).normalize()
        except Exception:
            return pd.NaT
    try:
        return pd.to_datetime(m.group(1)).normalize()
    except Exception:
        return pd.NaT
    
def standardize_workout_dates(w: pd.DataFrame) -> pd.DataFrame:
    w = w.copy()
    w["parsed_start"] = w["date"].astype(str).apply(parse_start_date)
    # Keep as Timestamp (normalized) to avoid mixing python date objects vs timestamps
    w["date"] = pd.to_datetime(w["parsed_start"]).dt.normalize()
    w = w.drop(columns=["parsed_start"])
    return w



In [26]:
workouts = standardize_workout_dates(workout)
workouts = workouts[workouts["date"] > pd.Timestamp("2024-11-01")]
workout_dates = set(workouts["date"].dropna().unique())
fitbit_dates = set(fitbit["date"].dropna().unique())
print(f"Workout dates: {len(workout_dates)}")
print(f"Fitbit dates:  {len(fitbit_dates)}")
missing_in_fitbit = workout_dates - fitbit_dates

print(f"Missing in Fitbit: {len(missing_in_fitbit)}")
print(f"Percent missing: {100 * len(missing_in_fitbit) / len(workout_dates):.2f}%")

Workout dates: 282
Fitbit dates:  516
Missing in Fitbit: 31
Percent missing: 10.99%


In [25]:
comparison = pd.DataFrame({
    "date": sorted(workout_dates | fitbit_dates)
})

comparison["in_workouts"] = comparison["date"].isin(workout_dates)
comparison["in_fitbit"] = comparison["date"].isin(fitbit_dates)

mismatches = comparison[
    comparison["in_workouts"] != comparison["in_fitbit"]
]

print(mismatches)

          date  in_workouts  in_fitbit
0   2022-06-01         True      False
1   2022-06-02         True      False
2   2022-06-06         True      False
3   2022-06-07         True      False
4   2022-06-08         True      False
..         ...          ...        ...
875 2026-05-12         True      False
876 2026-05-14         True      False
877 2026-05-20         True      False
878 2026-05-23         True      False
879 2026-05-24         True      False

[629 rows x 3 columns]
